# 18a — Controlled solver failures and repairs

**Prerequisites:** notebook 18 and first derivatives. **Outcome:** reproduce a
small failure, identify the relevant evidence, and compare a specific repair.
Run the core examples first; the derivative and rank sections are deeper checks.

These labs cover common failure classes, not every possible plugin/platform error.
Keep a minimal reproducer, versions, numeric inputs, full status, and residuals when
reporting a new issue. IPOPT solves local nonlinear problems; a successful solve
cannot certify global optimality or physical correctness of the model.


## Setup

The setup finds repository utilities and imports CasADi, NumPy, a timer, and shared
presentation helpers. Every numerical experiment runs as ordinary Python before
any optional widgets are created. This makes the lesson executable in CI and in a
static documentation build.


In [ ]:
# Locate the checkout without installing a model package.
import sys
from pathlib import Path

repository_root = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'casadi_tutorial').is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import numpy as np
from time import perf_counter

from casadi_tutorial.diagnostics import bound_report, kkt_residuals, solver_summary
from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_series


## 1. Preflight: shapes, bound consistency, parameters, and capabilities

A model-construction error happens before numerical convergence can be relevant.
The first operation below multiplies a $3\times3$ matrix by a length-two vector;
CasADi rejects the incompatible shapes. The second rejects a lower bound larger
than its upper bound, without starting a solver. Both are deliberate errors, and
an unexpected absence of an error fails the lesson rather than silently passing.


In [ ]:
def expect_preflight_error(label, operation, expected_type):
    try:
        operation()
    except expected_type as error:
        print(label, 'captured:', type(error).__name__, str(error).splitlines()[-1])
    else:
        raise AssertionError(f'{label} was expected to fail.')

expect_preflight_error(
    'incompatible shapes',
    lambda: ca.mtimes(ca.DM.eye(3), ca.MX.sym('wrong_length', 2)), RuntimeError,
)
expect_preflight_error(
    'inconsistent bounds', lambda: bound_report([0.0], [2.0], [1.0]), ValueError,
)
# Repair each input explicitly: dimensions and lower/upper ordering now agree.
assert ca.mtimes(ca.DM.eye(2), ca.DM([1, 2])).shape == (2, 1)
assert bound_report([0.0], [-1.0], [1.0])[0]['violation'] == 0


### An unset parameter is missing solve-time data

An Opti parameter is fixed data, not an additional variable to estimate. The
first solve below fails because `requested` has no value. Assigning it with
`set_value` repairs the same graph. In repeated estimation/control solves, assign
all current measurements and initial conditions before every call.


In [ ]:
parameter_problem = ca.Opti()
parameter_decision = parameter_problem.variable()
requested = parameter_problem.parameter()
parameter_problem.minimize((parameter_decision-requested)**2)
parameter_problem.solver('ipopt', {'print_time': False}, {'print_level': 0})
expect_preflight_error('unset parameter', parameter_problem.solve, RuntimeError)
parameter_problem.set_value(requested, 2.0)
parameter_solution = parameter_problem.solve()
print('assigned parameter solution:', parameter_solution.value(parameter_decision))
assert abs(parameter_solution.value(parameter_decision)-2.0) < 1e-8


### Check the compiled capability registry before choosing a plugin

The registry lists plugin names supplied by this CasADi build. A deliberately
absent name demonstrates the unavailable-plugin branch without attempting to load
an unknown binary. A listed name still may need external libraries or a license;
preserve a real initialization exception when those dependencies are missing.
Selecting another registered plugin requires checking that it supports your problem.


In [ ]:
registered_plugins = set(ca.CasadiMeta_plugins().split(';'))
requested_plugin = 'Nlpsol::teaching_solver_not_installed'
print('requested:', requested_plugin, 'registered:', requested_plugin in registered_plugins)
print('registered rootfinders:', sorted(name for name in registered_plugins
                                        if name.startswith('Rootfinder::')))
assert requested_plugin not in registered_plugins
# The environment's documented baseline is used by the other labs.
assert 'Nlpsol::ipopt' in registered_plugins
assert 'Rootfinder::newton' in registered_plugins
assert 'Integrator::cvodes' in registered_plugins


## 2. Scaling: compare equivalent physical and dimensionless models

Let $s=(10^{-4},10^4)$ and $z_i=x_i/s_i$. Minimize
$(z_0-1)^2+(z_1-2)^2$ subject to $z_0+z_1=2$. The exact solution is
$z^*=(0.5,1.5)$. In physical coordinates the Hessian is
$2\operatorname{diag}(1/s_i^2)$, with condition number $10^{16}$; in $z$ it is $2I$.

Both formulations below encode the same problem. IPOPT's automatic NLP scaling is
disabled **only for this comparison** so variable scaling is visible. Judge a result
by dimensionless physical residuals and the solution, not by an assumed portable
iteration count or time improvement.


In [ ]:
scales = np.array([1e-4, 1e4])

def scaling_case(dimensionless):
    decision = ca.MX.sym('scaled_decision', 2)
    normalized = decision if dimensionless else decision / ca.DM(scales)
    objective = (normalized[0] - 1)**2 + (normalized[1] - 2)**2
    constraint = normalized[0] + normalized[1]
    solver = ca.nlpsol('scaling_case', 'ipopt',
                       {'x': decision, 'f': objective, 'g': constraint},
                       {'print_time': False, 'error_on_fail': False,
                        'ipopt.print_level': 0, 'ipopt.nlp_scaling_method': 'none',
                        'ipopt.max_iter': 100, 'ipopt.tol': 1e-10})
    before = perf_counter()
    result = solver(x0=[0, 0], lbg=2, ubg=2)
    elapsed = perf_counter() - before
    candidate = np.asarray(result['x'].full()).ravel()
    z_value = candidate if dimensionless else candidate / scales
    return z_value, solver.stats(), elapsed

raw_z, raw_stats, raw_time = scaling_case(False)
scaled_z, scaled_stats, scaled_time = scaling_case(True)
print('Hessian condition numbers:', np.linalg.cond(np.diag(2/scales**2)), 1.0)
for name, point, stats, elapsed in [
    ('physical coordinates', raw_z, raw_stats, raw_time),
    ('dimensionless coordinates', scaled_z, scaled_stats, scaled_time),
]:
    print(name, solver_summary(stats, elapsed_s=elapsed),
          'normalized solution:', point, 'equality residual:', abs(point.sum()-2))
assert scaled_stats['success']
np.testing.assert_allclose(scaled_z, [0.5, 1.5], atol=1e-7)


### Interpret the scaling trace

The solver may solve both versions quickly on your machine. Conditioning still
predicts sensitivity of the linear algebra; it does not guarantee a particular
failure or iteration reduction. These curves report IPOPT's own residual measures.
A plotting floor avoids logarithms of zero and affects presentation only.


In [ ]:
for name, stats in [('physical', raw_stats), ('dimensionless', scaled_stats)]:
    history = stats.get('iterations', {})
    curves = {key: np.maximum(history[key], 1e-16)
              for key in ('inf_pr', 'inf_du') if key in history}
    if curves:
        display_plot(plot_series(np.arange(len(next(iter(curves.values())))), curves,
                                 title=f'Scaling comparison: {name}', xlabel='iteration',
                                 ylabel='reported infeasibility', log_y=True))


## 3. Domain failures can occur in derivatives alone

For $f(v)=\sqrt{v-1}$, $v<1$ is outside the real domain. At $v=1$ the value is
finite but $f'(v)=1/(2\sqrt{v-1})$ is infinite. Checking the objective alone misses
this boundary problem. `regularity_check=True` raises for invalid numeric outputs,
so the loop distinguishes an invalid value from an invalid derivative.

For a physically nonnegative root $r$, the exact parameterization $v=1+r^2$,
$r\ge0$, gives $\sqrt{v-1}=r$. Formulating an objective directly in $r$ can remove
the singular derivative without inventing a hidden epsilon. This is useful only
when that parameterization respects the intended physical problem.


In [ ]:
v = ca.MX.sym('domain_variable')
root_expression = ca.sqrt(v - 1)
value_check = ca.Function('root_value', [v], [root_expression], {'regularity_check': True})
derivative_check = ca.Function('root_derivative', [v], [ca.gradient(root_expression, v)],
                               {'regularity_check': True})
for point in [0.0, 1.0, 2.0]:
    for label, function in [('value', value_check), ('derivative', derivative_check)]:
        try:
            print(point, label, float(function(point)))
        except RuntimeError:
            print(point, label, 'INVALID: inspect domain before tuning options')
r = ca.MX.sym('nonnegative_root')
repaired_solver = ca.nlpsol('root_parameterization', 'ipopt',
                           {'x': r, 'f': (r - 1)**2},
                           {'print_time': False, 'ipopt.print_level': 0})
repaired = repaired_solver(x0=0.2, lbx=0)
print('repaired root:', float(repaired['x']), 'physical v:', 1+float(repaired['x'])**2)
assert repaired_solver.stats()['success']
assert abs(float(repaired['x']) - 1) < 1e-6


## 4. Deeper check: verify derivative values along a direction

For smooth $F$, a centered finite difference approximates
$J_F(x)d\approx[F(x+hd)-F(x-hd)]/(2h)$. Compare several step sizes: truncation
dominates large $h$ and cancellation dominates extremely small $h$. Keep perturbed
points inside the function's domain. The deliberately doubled derivative below is
wrong and should be distinguishable from AD.

AD differentiates the expression you wrote; agreement cannot reveal wrong units,
a missing physical term, or an incorrect data model. IPOPT's `derivative_test`
option is most useful for callbacks, external functions, and custom derivatives.


In [ ]:
z = ca.MX.sym('derivative_point', 2)
expression = ca.vertcat(ca.sin(z[0]) + z[1]**2, ca.exp(0.2*z[0])*z[1])
function = ca.Function('derivative_values', [z], [expression])
jacobian = ca.Function('derivative_jacobian', [z], [ca.jacobian(expression, z)])
point, direction = np.array([0.4, 0.7]), np.array([0.6, -0.8])
ad_direction = np.asarray((jacobian(point) @ direction).full()).ravel()
steps = np.logspace(-2, -8, 13)
correct_error, wrong_error = [], []
for step in steps:
    difference = np.asarray(((function(point+step*direction)-function(point-step*direction))
                             / (2*step)).full()).ravel()
    correct_error.append(np.linalg.norm(difference-ad_direction, ord=np.inf))
    wrong_error.append(np.linalg.norm(difference-2*ad_direction, ord=np.inf))
display_plot(plot_series(steps, {'AD versus finite difference': correct_error,
                                'incorrect derivative': wrong_error},
                         xlabel='finite-difference step', ylabel='directional error',
                         title='Derivative check', log_y=True))
assert min(correct_error) < 1e-7
assert min(wrong_error) > 0.1


## 5. Deeper check: distinguish redundancy from a difficult starting point

The rows $g_0=x_0+x_1-1$ and $g_1=2g_0$ enforce the same equality. Their Jacobian
has two rows but rank one; here the exact algebra proves redundancy everywhere.
At a single point, rank loss alone could instead be local (for example, the
Jacobian of $x^2=0$ at zero). Remove a truly redundant constraint rather than
loosening tolerances. Singular values provide a numerical clue, not a proof of
global redundancy in arbitrary nonlinear models.


In [ ]:
rank_x = ca.MX.sym('rank_x', 2)
base_constraint = rank_x[0] + rank_x[1] - 1
redundant = ca.vertcat(base_constraint, 2*base_constraint)
rank_jacobian = ca.Function('rank_jacobian', [rank_x], [ca.jacobian(redundant, rank_x)])
matrix = np.asarray(rank_jacobian([0, 0]).full())
print('Jacobian:', matrix, 'singular values:', np.linalg.svd(matrix, compute_uv=False))
assert np.linalg.matrix_rank(matrix) == 1
rank_solver = ca.nlpsol('independent_rows', 'ipopt',
                       {'x': rank_x, 'f': ca.sumsqr(rank_x-ca.DM([0.2, 0.8])),
                        'g': base_constraint},
                       {'print_time': False, 'ipopt.print_level': 0})
rank_result = rank_solver(x0=[0, 0], lbg=0, ubg=0)
print('repaired independent-row solution:', rank_result['x'])
assert rank_solver.stats()['success']


## 6. Nonsmoothness: make the modeling tradeoff explicit

The left and right slopes of $|x|$ at zero differ, even if a software convention
returns zero as a derivative there. Smooth NLP solvers assume suitable smoothness.
The approximation $\sqrt{x^2+\epsilon^2}$ is smooth but changes the objective; its
error is at most $\epsilon$. A physically exact reformulation or a solver suited
to the nonsmooth problem may be preferable. Never call smoothing an exact repair.


In [ ]:
mesh = np.linspace(-0.05, 0.05, 301)
epsilon = 0.01
left_slope = (abs(0.0)-abs(-1e-6))/1e-6
right_slope = (abs(1e-6)-abs(0.0))/1e-6
print('left/right slopes at zero:', left_slope, right_slope)
display_plot(plot_series(mesh, {'absolute value': abs(mesh),
                                'smooth approximation': np.sqrt(mesh**2+epsilon**2)},
                         xlabel='x', ylabel='objective term', title='Smoothing changes the model'))
assert left_slope != right_slope
assert np.max(np.sqrt(mesh**2+epsilon**2)-abs(mesh)) <= epsilon + 1e-14


## 7. Iteration limit: inspect progress before changing the budget

A one-iteration budget intentionally stops an ordinary bounded Rosenbrock solve.
The identical model with a larger budget serves as a controlled comparison.
`error_on_fail=False` retains the failed result for diagnosis; only the successful
result is accepted. On a different problem, simply raising this limit may hide
poor scaling, invalid derivatives, or an infeasible request.


In [ ]:
budget_x = ca.MX.sym('budget_x', 2)
budget_f = (1-budget_x[0])**2 + 100*(budget_x[1]-budget_x[0]**2)**2

def budget_case(iterations):
    solver = ca.nlpsol('iteration_budget', 'ipopt', {'x': budget_x, 'f': budget_f},
                       {'print_time': False, 'error_on_fail': False,
                        'ipopt.print_level': 0, 'ipopt.max_iter': int(iterations)})
    result = solver(x0=[-1.2, 1], lbx=-2, ubx=2)
    return result, solver.stats()

limited, limited_stats = budget_case(1)
recovered, recovered_stats = budget_case(150)
print('limited:', solver_summary(limited_stats), 'objective:', float(limited['f']))
print('recovered:', solver_summary(recovered_stats), 'objective:', float(recovered['f']))
assert not limited_stats['success']
assert recovered_stats['success'] and float(recovered['f']) < 1e-10
history = recovered_stats.get('iterations', {})
if 'inf_du' in history:
    display_plot(plot_series(np.arange(len(history['inf_du'])),
                             {'dual infeasibility': np.maximum(history['inf_du'], 1e-16)},
                             xlabel='iteration', ylabel='reported dual infeasibility',
                             title='Convergence with sufficient budget', log_y=True))


## 8. Initialization and local minima

For $f(x)=(x^2-1)^2+0.1x$ on $[-2,2]$, starts near $-1$ and $+1$ converge to
different stationary minima. Both solves can report success. Compare objective
values and feasibility, and inspect multiple starts when the problem is nonconvex.
Two starts do not constitute a global-optimality certificate.


In [ ]:
local_x = ca.MX.sym('local_x')
local_objective = (local_x**2-1)**2 + 0.1*local_x
local_solver = ca.nlpsol('local_minima', 'ipopt', {'x': local_x, 'f': local_objective},
                        {'print_time': False, 'ipopt.print_level': 0})
local_results = []
for start in [-1, 1]:
    result = local_solver(x0=start, lbx=-2, ubx=2)
    assert local_solver.stats()['success']
    local_results.append((float(result['x']), float(result['f'])))
print('different accepted local candidates:', local_results)
grid = np.linspace(-1.5, 1.5, 301)
display_plot(plot_series(grid, {'objective': (grid**2-1)**2+0.1*grid},
                         xlabel='decision', ylabel='objective', title='A successful local solution'))
assert local_results[1][1] - local_results[0][1] > 0.1


## 9. Infeasibility analysis with explicit slacks

The inconsistent request $x\ge3$, $x\le1$ can be diagnosed by minimizing
$s_0^2+s_1^2$ subject to $x+s_0\ge3$, $x-s_1\le1$, $s_i\ge0$.
The optimal violations are one unit on each side. This phase-I problem explains
which requests must relax; its solution is **not feasible for the original problem**.
Slack units, relative weights, and allowable violations must have physical meaning
before using soft constraints in an application.


In [ ]:
phase = ca.Opti()
phase_x = phase.variable()
slack = phase.variable(2)
phase.minimize(ca.sumsqr(slack))
phase.subject_to(slack >= 0)
phase.subject_to(phase_x+slack[0] >= 3)
phase.subject_to(phase_x-slack[1] <= 1)
phase.set_initial(phase_x, 2)
phase.set_initial(slack, 1)
phase.solver('ipopt', {'print_time': False}, {'print_level': 0})
phase_solution = phase.solve()
print('phase-I x:', phase_solution.value(phase_x),
      'required slacks:', phase_solution.value(slack))
np.testing.assert_allclose(phase_solution.value(slack), [1, 1], atol=1e-6)


## 10. A rootfinder can return a poor candidate after its iteration limit

Solve $r(z)=z^2-2=0$ from $z_0=10$ with Newton's method. One iteration yields
$z_1=5.1$, whose residual is still large. Retaining that point is useful for
diagnosis but not acceptance. The repaired run keeps the same residual and initial
point and supplies enough iterations. Always check the residual independently:
a small step is not necessarily a small root residual, especially near singular
Jacobians. A starting point near zero would also make this example's derivative
$2z$ problematic.


In [ ]:
root_variable = ca.MX.sym('root_variable')
root_residual = ca.Function('root_residual', [root_variable], [root_variable**2-2])
limited_rootfinder = ca.rootfinder('limited_rootfinder', 'newton', root_residual,
                                 {'max_iter': 1, 'error_on_fail': False})
limited_root = limited_rootfinder(10.0)
print('limited root:', float(limited_root), solver_summary(limited_rootfinder.stats()),
      'residual:', float(root_residual(limited_root)))
assert not limited_rootfinder.stats()['success']
assert abs(float(root_residual(limited_root))) > 1
repaired_rootfinder = ca.rootfinder('repaired_rootfinder', 'newton', root_residual,
                                  {'max_iter': 30, 'abstol': 1e-12})
repaired_root = repaired_rootfinder(10.0)
print('repaired root:', float(repaired_root), solver_summary(repaired_rootfinder.stats()),
      'residual:', float(root_residual(repaired_root)))
assert repaired_rootfinder.stats()['success']
assert abs(float(root_residual(repaired_root))) < 1e-10


## 11. An integrator resource limit is different from an invalid model

For $\dot x=-1000x$, $x(0)=1$, the exact solution is $e^{-1000t}$. CVODES must
resolve a rapid initial transient. A cap of one internal step deliberately prevents
it from reaching $t=0.02$ and raises an integration error. The recovered run uses
the same model, horizon, and tolerances with a larger step cap, then checks against
the analytic solution. This controlled resource-limit example does not imply that
raising a cap repairs domain errors, inconsistent DAE initial conditions, or a
wrong differential equation. CVODES writes an expected diagnostic to stderr below.


In [ ]:
stiff_state = ca.MX.sym('stiff_state')
stiff_problem = {'x': stiff_state, 'ode': -1000*stiff_state}
integrator_options = {'abstol': 1e-10, 'reltol': 1e-8}
limited_integrator = ca.integrator('limited_integrator', 'cvodes', stiff_problem, 0, 0.02,
                                  {**integrator_options, 'max_num_steps': 1})
expect_preflight_error('integration step limit', lambda: limited_integrator(x0=1), RuntimeError)
repaired_integrator = ca.integrator('repaired_integrator', 'cvodes', stiff_problem, 0, 0.02,
                                   {**integrator_options, 'max_num_steps': 10000})
terminal = float(repaired_integrator(x0=1)['xf'])
print('integrated final state:', terminal, 'exact:', np.exp(-20),
      'internal steps:', repaired_integrator.stats().get('nsteps'))
assert np.isfinite(terminal) and abs(terminal-np.exp(-20)) < 1e-9


## 12. Run a controlled budget experiment

Change the budget, predict whether the solver will finish, and run once. Reset
restores the control without rerunning. The deterministic limited/recovered cases
above execute even when widgets are disabled. A successful result still needs the
physical feasibility checks used in notebook 18.


In [ ]:
def inspect_budget(iterations):
    result, stats = budget_case(iterations)
    print(solver_summary(stats), 'candidate:', result['x'], 'objective:', float(result['f']))
    trace = stats.get('iterations', {})
    if 'inf_du' in trace:
        display_plot(plot_series(np.arange(len(trace['inf_du'])),
                                 {'dual infeasibility': np.maximum(trace['inf_du'], 1e-16)},
                                 xlabel='iteration', ylabel='reported dual infeasibility',
                                 title='Budget experiment', log_y=True))

experiment(inspect_budget, {'iterations': 1}, title='Diagnose an iteration budget')


## Recap and extension exercises

1. Explain why a finite square-root value can still break a solver derivative.
2. Add a third redundant equality; distinguish row count from rank.
3. Compare scaling using the same physical initial point and acceptance residuals.
4. Give a case where smoothing changes the best action enough to matter.
5. Explain why the phase-I result does not repair the original request.

For external derivatives, try IPOPT options `derivative_test='first-order'` and
`check_derivatives_for_naninf='yes'` on a small model before scaling it up. Keep
verbose output bounded and preserve the full failing log. Linear-solver availability
and plugin initialization errors require inspecting installed plugins and the
specific exception; they are different from failure of an initialized NLP solve.

References: [CasADi Function options and debugging](https://web.casadi.org/docs/),
[IPOPT derivative checker and diagnostic options](https://coin-or.github.io/Ipopt/OPTIONS.html).
